标准Torch训练流程

全连接NN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 1. 定义模型（继承nn.Module）
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)  # 输入层
        self.fc2 = nn.Linear(128, 10)   # 输出层

    def forward(self, x):
        x = x.view(-1, 784)
        #print(self.fc1(x))
        x = torch.relu(self.fc1(x))     # 激活函数
        #print(x)
        return self.fc2(x)

# 2. 数据准备
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='../data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 3. 实例化模型、损失函数和优化器
model = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. 训练循环
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
for epoch in range(2):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        #print(data)
        optimizer.zero_grad()          # 清零梯度
        output = model(data)           # 前向传播
        loss = criterion(output, target)  # 计算损失
        loss.backward()                # 反向传播
        optimizer.step()               # 更新参数
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')
    
# 5. 验证部分：输入一个测试样本并给出输出
test_dataset = datasets.MNIST(root='../data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)  # 单样本加载

# 取第一个测试样本（您可修改为特定索引）
data, target = next(iter(test_loader))
data, target = data.to(device), target.to(device)

model.eval()  # 切换到评估模式
with torch.no_grad():
    output = model(data)
    pred = output.argmax(dim=1, keepdim=True).item()  # 预测标签
    print(f'Input Label (True): {target.item()}')
    print(f'Predicted Label: {pred}')
    print(f'Output Logits: {output.squeeze().tolist()}')  # 完整输出向量

# 将张量转换为可显示的格式
img_to_show = data.cpu().squeeze(0).permute(1, 2, 0)  # (C,H,W) -> (H,W,C)
#plt.imshow(img_to_show.squeeze(), cmap='gray')        # 灰度图处理
plt.imshow(img_to_show.squeeze())  # 原图
plt.axis('off')  # 隐藏坐标轴
plt.savefig('./output/input_image.png', bbox_inches='tight')  # 保存文件
plt.show()
plt.close()  # 关闭图形避免重复显示
torch.save(model.state_dict(), "nn_full.pth")
print(f'Input image saved to: ./output/input_image.png')

CNN

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# 1. 定义模型（继承nn.Module）
class SimpleNet(nn.Module):
    def __init__(self):
        super(SimpleNet, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, padding=1)  # 输入1通道，输出32通道，3x3卷积核
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)  # 2x2池化
        self.fc1 = nn.Linear(64 * 7 * 7, 128)  # 全连接层
        self.fc2 = nn.Linear(128, 10)  # 输出层，10个类别
        self.dropout = nn.Dropout(0.5)  # Dropout层
        self.relu = nn.ReLU()  # ReLU激活函数

    def forward(self, x):
        # 卷积层1 -> ReLU -> 池化
        x = self.pool(self.relu(self.conv1(x)))
        # 卷积层2 -> ReLU -> 池化
        x = self.pool(self.relu(self.conv2(x)))
        # 展平特征图
        x = x.view(-1, 64 * 7 * 7)
        # 全连接层1 -> ReLU -> Dropout
        x = self.dropout(self.relu(self.fc1(x)))
        # 输出层
        x = self.fc2(x)
        return x

# 2. 数据准备
transform = transforms.ToTensor()
train_dataset = datasets.MNIST(root='../data', train=True, download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# 3. 实例化模型、损失函数和优化器
model = SimpleNet()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. 训练循环
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
for epoch in range(2):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()          # 清零梯度
        output = model(data)           # 前向传播
        loss = criterion(output, target)  # 计算损失
        loss.backward()                # 反向传播
        optimizer.step()               # 更新参数
    print(f'Epoch {epoch+1}, Loss: {loss.item()}')
    
# 5. 验证部分：输入一个测试样本并给出输出
test_dataset = datasets.MNIST(root='../data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=True)  # 单样本加载

# 取第一个测试样本（您可修改为特定索引）
data, target = next(iter(test_loader))
data, target = data.to(device), target.to(device)

model.eval()  # 切换到评估模式
with torch.no_grad():
    output = model(data)
    pred = output.argmax(dim=1, keepdim=True).item()  # 预测标签
    print(f'Input Label (True): {target.item()}')
    print(f'Predicted Label: {pred}')
    print(f'Output Logits: {output.squeeze().tolist()}')  # 完整输出向量

# 将张量转换为可显示的格式
img_to_show = data.cpu().squeeze(0).permute(1, 2, 0)  # (C,H,W) -> (H,W,C)
#plt.imshow(img_to_show.squeeze(), cmap='gray')        # 灰度图处理
plt.imshow(img_to_show.squeeze())  # 原图
plt.axis('off')  # 隐藏坐标轴
plt.savefig('./output/input_image.png', bbox_inches='tight')  # 保存文件
plt.show()
plt.close()  # 关闭图形避免重复显示
print(f'Input image saved to: ./output/input_image.png')

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from torch.utils.data import DataLoader

# 定义变换：将PIL图像转换为张量
transform = transforms.Compose([transforms.ToTensor()])

# 加载CIFAR-10训练数据集（会自动下载，如果本地没有）
trainset = torchvision.datasets.CIFAR10(root='../data', train=True, download=True, transform=transform)
trainloader = DataLoader(trainset, batch_size=100, shuffle=False, num_workers=2)

# 加载CIFAR-10测试数据集
testset = torchvision.datasets.CIFAR10(root='../data', train=False, download=True, transform=transform)
testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

classes = testset.classes  # 返回列表：['airplane', 'automobile', ...]
print(classes)

# CIFAR-10的类别标签
#classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# 函数：展示图像网格
def imshow(img):
    #img = img / 2 + 0.5  # 反归一化（如果需要）
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))
    plt.axis('off')

# 注意：CIFAR-10总共有60,000张图像（50,000训练 + 10,000测试）。
# 直接展示所有图像在单个窗口中不现实，因为图像太多，会导致内存问题或显示混乱。
# 建议分批展示，或者展示一个样本网格。下面展示如何分批处理所有图像。

# 示例：展示训练集的前100张图像作为一个网格
dataiter = iter(trainloader)
images, labels = next(dataiter)

# 创建一个4x25的网格（100张图像）
fig = plt.figure(figsize=(25, 4))
for idx in range(100):
    ax = fig.add_subplot(4, 25, idx + 1, xticks=[], yticks=[])
    imshow(images[idx])
    ax.set_title(classes[labels[idx]])

plt.show()  # 在你的本地环境中运行此代码会显示图像
plt.close

# 要展示所有图像：你可以循环遍历整个dataloader，并在多个figure中展示，但这会生成许多窗口。
# 例如，展示整个训练集（分批）：
"""
for batch_idx, (images, labels) in enumerate(trainloader):
    fig = plt.figure(figsize=(25, 4))
    for idx in range(len(images)):
        ax = fig.add_subplot(4, 25, idx + 1, xticks=[], yticks=[])
        imshow(images[idx])
        ax.set_title(classes[labels[idx]])
    plt.show()  # 每个批次显示一个窗口，关闭后显示下一个
"""

# 类似地，对测试集重复以上操作。
# 注意：在实际运行中，这会弹出500个窗口（每个批次100张，训练集500批），非常不实用。
# 推荐使用更高效的方式，如保存到文件或使用tensorboard等工具查看。